# pisto-gpt-64m — Colab Training (Drive-backed)

This notebook trains the model on a Colab T4 GPU and **saves weights to your Google Drive every 30 seconds**, so a VM recycle never loses your progress.

## Setup (one time)
1. Upload `pisto-gpt-64m.tgz` to your Drive root (`MyDrive/pisto-gpt-64m.tgz`)
2. Run all cells below

## After a VM death
Just re-run all cells — training **resumes** from the last checkpoint in Drive.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# 2. Extract the repo from Drive
import os, subprocess, shutil

BUNDLE = '/content/drive/MyDrive/pisto-gpt-64m.tgz'
REPO   = '/content/pg'
DRIVE_W = '/content/drive/MyDrive/pg-weights'

assert os.path.exists(BUNDLE), 'Upload pisto-gpt-64m.tgz to MyDrive first!'

os.makedirs(REPO, exist_ok=True)
os.makedirs(DRIVE_W, exist_ok=True)

subprocess.run(f'cd {REPO} && tar xzf "{BUNDLE}"', shell=True, check=True)

# Restore last checkpoint from Drive (if any) so training resumes
os.makedirs(f'{REPO}/weights', exist_ok=True)
for f in ['best.pt', 'log.jsonl']:
    src = f'{DRIVE_W}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{REPO}/weights/{f}')
        print(f'Restored {f} from Drive')

print('Repo ready at', REPO)

In [ ]:
# 3. Install dependencies
!pip install -q torch datasets tokenizers
print('Deps installed')

In [ ]:
# 4. Start training in the background
import subprocess, threading, time, shutil, os, json

REPO   = '/content/pg'
DRIVE_W = '/content/drive/MyDrive/pg-weights'

# Launch pretraining as a background process
logf = open(f'{REPO}/train.out', 'w')
proc = subprocess.Popen(
    ['python3', 'training/pretrain.py'],
    cwd=REPO, stdout=logf, stderr=subprocess.STDOUT
)
print(f'Training started (PID {proc.pid})')

# Background thread: sync weights to Drive every 30s
def sync_to_drive():
    while True:
        try:
            for f in ['best.pt', 'log.jsonl']:
                src = f'{REPO}/weights/{f}'
                if os.path.exists(src):
                    shutil.copy(src, f'{DRIVE_W}/{f}')
        except Exception as e:
            print('sync error:', e)
        time.sleep(30)

threading.Thread(target=sync_to_drive, daemon=True).start()
print('Drive sync active (every 30s) — weights are safe')

In [ ]:
# 5. Watch training progress (re-run this cell anytime)
import subprocess
print(subprocess.run(['tail', '-15', '/content/pg/train.out'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 6. Check what's saved in Drive
import os
DRIVE_W = '/content/drive/MyDrive/pg-weights'
for f in sorted(os.listdir(DRIVE_W)):
    p = os.path.join(DRIVE_W, f)
    print(f'{f:15s} {os.path.getsize(p)/1e6:.1f} MB')
print('\nbest.pt in Drive = your safe checkpoint')